## Criterio de inclusión distancia <= 2 vóxeles

### Dataset 111 2d 250 epochs

In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label

# ========================= RUTAS =========================
EXCEL_PATH = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/CSIRO_dataset/rCMBInformationInfo.xlsx"
MASK_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results/Dataset111_SyntheticCMB/nnUNet_results_test_with_rCMBs" 
OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results/Dataset111_SyntheticCMB/analisis_prediccions_sobre_rCMBs/Trial1_Dataset111_2D"


os.makedirs(OUTPUT_DIR, exist_ok=True)

def safe_int_minus1(val):
    if pd.isna(val) or str(val).strip() == '': return None
    try:
        raw = int(float(str(val).strip()))
        return raw - 1 if raw >= 0 else None # Ajustado para permitir index 0
    except: return None

# ========================= PROCESAMIENTO =========================
df = pd.read_excel(EXCEL_PATH)
results_list = []
total_rcmbs_real_global = 0

for idx, row in df.iterrows():
    nifti_name = str(row.iloc[0]).strip()
    if not nifti_name.endswith('.nii.gz'): continue

    mask_path = os.path.join(MASK_DIR, nifti_name)
    if not os.path.exists(mask_path): continue

    mask_data = nib.load(mask_path).get_fdata()
    shape = mask_data.shape

    # 1. Extraer coordenadas reales (Ground Truth)
    coords_gt = []
    for i in range(1, len(row), 3):
        x, y, z = safe_int_minus1(row.iloc[i]), safe_int_minus1(row.iloc[i+1]), safe_int_minus1(row.iloc[i+2])
        if all(v is not None for v in [x, y, z]):
            if 0 <= x < shape[0] and 0 <= y < shape[1] and 0 <= z < shape[2]:
                coords_gt.append((x, y, z))
    
    total_rcmbs_real_global += len(coords_gt)

    # 2. Etiquetar detecciones del modelo (sCMBs detectados)
    labeled_mask, num_sCMB_total = label(mask_data > 0, return_num=True)
    
    # 3. Verificación con vecindario (Distancia Euclídea <= 2)
    
    # [Added by me: Generación precalculada de offsets esféricos para optimizar el bucle]
    # Busca en un rango de -2 a 2 en cada eje, pero solo guarda los vóxeles cuya
    # distancia euclídea al centro sea <= 2.
    valid_offsets = []
    for dx in range(-2, 3):
        for dy in range(-2, 3):
            for dz in range(-2, 3):
                if (dx**2 + dy**2 + dz**2) <= 4:
                    valid_offsets.append((dx, dy, dz))

    blobs_hit = set()
    tp_count = 0
    
    for (cx, cy, cz) in coords_gt:
        found_in_neighborhood = False
        
        # Iterar únicamente sobre los vóxeles que cumplen el criterio esférico
        for dx, dy, dz in valid_offsets:
            nx, ny, nz = cx + dx, cy + dy, cz + dz
            
            # Verificar que no nos salimos de los límites de la imagen
            if 0 <= nx < shape[0] and 0 <= ny < shape[1] and 0 <= nz < shape[2]:
                blob_id = labeled_mask[nx, ny, nz]
                if blob_id > 0:
                    blobs_hit.add(blob_id)
                    found_in_neighborhood = True
                    # Nota: No ponemos 'break' aquí para que registre todos los blobs 
                    # distintos que pudieran estar tocando este radio, igual que en tu código original.
                    
        if found_in_neighborhood:
            tp_count += 1

    # Cálculos de sCMB (Detecciones del modelo)
    scmb_correctos = len(blobs_hit) 
    scmb_erroneos = num_sCMB_total - scmb_correctos
    fn = len(coords_gt) - tp_count

    # Métricas
    # [Added by me: Prevención de división por cero manejada explícitamente en tu código original, la mantengo igual]
    precision = tp_count / num_sCMB_total if num_sCMB_total > 0 else 0
    recall = tp_count / len(coords_gt) if len(coords_gt) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    results_list.append({
        'ID': nifti_name,
        'rCMBs_Reales': len(coords_gt),
        'sCMBs_Detectados_Total': num_sCMB_total,
        'sCMBs_Correctos(TP)': scmb_correctos,
        'sCMBs_Erroneos(FP)': scmb_erroneos,
        'Missed(FN)': fn,
        'Precision': round(precision, 4),
        'Recall': round(recall, 4),
        'F1-Score': round(f1, 4)
    })



# ========================= SALIDA =========================
df_res = pd.DataFrame(results_list)
csv_path = os.path.join(OUTPUT_DIR, "analisis_completo_CMB.csv") # distancia <= 2 vóxeles
df_res.to_csv(csv_path, index=False)

total_scmb_gen = df_res['sCMBs_Detectados_Total'].sum()
total_scmb_corr = df_res['sCMBs_Correctos(TP)'].sum()
total_scmb_err = df_res['sCMBs_Erroneos(FP)'].sum()

print("-" * 40)
print(f"ANÁLISIS FINALIZADO")
print(f"Total rCMBs (Reales en Excel): {total_rcmbs_real_global}")
print(f"Total sCMBs (Generados por el modelo): {total_scmb_gen}")
print(f"  > sCMBs Correctos (TP): {total_scmb_corr}")
print(f"  > sCMBs Erróneos (FP): {total_scmb_err}")
print("-" * 40)
print(f"Media Recall (Sensibilidad): {df_res['Recall'].mean():.4f}")
print(f"Media Precision: {df_res['Precision'].mean():.4f}")
print(f"Media F1-Score: {df_res['F1-Score'].mean():.4f}")
print(f"Resultados guardados en: {csv_path}")

----------------------------------------
ANÁLISIS FINALIZADO
Total rCMBs (Reales en Excel): 146
Total sCMBs (Generados por el modelo): 110
  > sCMBs Correctos (TP): 62
  > sCMBs Erróneos (FP): 48
----------------------------------------
Media Recall (Sensibilidad): 0.4237
Media Precision: 0.4718
Media F1-Score: 0.4210
Resultados guardados en: /media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results/Dataset111_SyntheticCMB/analisis_prediccions_sobre_rCMBs/distancia_2vox/analisis_completo_CMB.csv


### Dataset 112 2d 250 epochs

In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label

# ========================= RUTAS =========================
EXCEL_PATH = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/CSIRO_dataset/rCMBInformationInfo.xlsx"
MASK_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results/Dataset112_SyntheticCMB/nnUNet_2D_250epochs_predicciones_test_rCMBs" 
OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results/Dataset112_SyntheticCMB/analisis_predicciones_sobre_rCMBs/"


os.makedirs(OUTPUT_DIR, exist_ok=True)

def safe_int_minus1(val):
    if pd.isna(val) or str(val).strip() == '': return None
    try:
        raw = int(float(str(val).strip()))
        return raw - 1 if raw >= 0 else None # Ajustado para permitir index 0
    except: return None

# ========================= PROCESAMIENTO =========================
df = pd.read_excel(EXCEL_PATH)
results_list = []
total_rcmbs_real_global = 0

for idx, row in df.iterrows():
    nifti_name = str(row.iloc[0]).strip()
    if not nifti_name.endswith('.nii.gz'): continue

    mask_path = os.path.join(MASK_DIR, nifti_name)
    if not os.path.exists(mask_path): continue

    mask_data = nib.load(mask_path).get_fdata()
    shape = mask_data.shape

    # 1. Extraer coordenadas reales (Ground Truth)
    coords_gt = []
    for i in range(1, len(row), 3):
        x, y, z = safe_int_minus1(row.iloc[i]), safe_int_minus1(row.iloc[i+1]), safe_int_minus1(row.iloc[i+2])
        if all(v is not None for v in [x, y, z]):
            if 0 <= x < shape[0] and 0 <= y < shape[1] and 0 <= z < shape[2]:
                coords_gt.append((x, y, z))
    
    total_rcmbs_real_global += len(coords_gt)

    # 2. Etiquetar detecciones del modelo (sCMBs detectados)
    labeled_mask, num_sCMB_total = label(mask_data > 0, return_num=True)
    
    # 3. Verificación con vecindario (Distancia Euclídea <= 2)
    
    # [Added by me: Generación precalculada de offsets esféricos para optimizar el bucle]
    # Busca en un rango de -2 a 2 en cada eje, pero solo guarda los vóxeles cuya
    # distancia euclídea al centro sea <= 2.
    valid_offsets = []
    for dx in range(-2, 3):
        for dy in range(-2, 3):
            for dz in range(-2, 3):
                if (dx**2 + dy**2 + dz**2) <= 4:
                    valid_offsets.append((dx, dy, dz))

    blobs_hit = set()
    tp_count = 0
    
    for (cx, cy, cz) in coords_gt:
        found_in_neighborhood = False
        
        # Iterar únicamente sobre los vóxeles que cumplen el criterio esférico
        for dx, dy, dz in valid_offsets:
            nx, ny, nz = cx + dx, cy + dy, cz + dz
            
            # Verificar que no nos salimos de los límites de la imagen
            if 0 <= nx < shape[0] and 0 <= ny < shape[1] and 0 <= nz < shape[2]:
                blob_id = labeled_mask[nx, ny, nz]
                if blob_id > 0:
                    blobs_hit.add(blob_id)
                    found_in_neighborhood = True
                    # Nota: No ponemos 'break' aquí para que registre todos los blobs 
                    # distintos que pudieran estar tocando este radio, igual que en tu código original.
                    
        if found_in_neighborhood:
            tp_count += 1

    # Cálculos de sCMB (Detecciones del modelo)
    scmb_correctos = len(blobs_hit) 
    scmb_erroneos = num_sCMB_total - scmb_correctos
    fn = len(coords_gt) - tp_count

    # Métricas
    # [Added by me: Prevención de división por cero manejada explícitamente en tu código original, la mantengo igual]
    precision = tp_count / num_sCMB_total if num_sCMB_total > 0 else 0
    recall = tp_count / len(coords_gt) if len(coords_gt) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    results_list.append({
        'ID': nifti_name,
        'rCMBs_Reales': len(coords_gt),
        'sCMBs_Detectados_Total': num_sCMB_total,
        'sCMBs_Correctos(TP)': scmb_correctos,
        'sCMBs_Erroneos(FP)': scmb_erroneos,
        'Missed(FN)': fn,
        'Precision': round(precision, 4),
        'Recall': round(recall, 4),
        'F1-Score': round(f1, 4)
    })



# ========================= SALIDA =========================
df_res = pd.DataFrame(results_list)
csv_path = os.path.join(OUTPUT_DIR, "analisis_completo_CMB.csv")
df_res.to_csv(csv_path, index=False)

total_scmb_gen = df_res['sCMBs_Detectados_Total'].sum()
total_scmb_corr = df_res['sCMBs_Correctos(TP)'].sum()
total_scmb_err = df_res['sCMBs_Erroneos(FP)'].sum()

print("-" * 40)
print(f"ANÁLISIS FINALIZADO")
print(f"Total rCMBs (Reales en Excel): {total_rcmbs_real_global}")
print(f"Total sCMBs (Generados por el modelo): {total_scmb_gen}")
print(f"  > sCMBs Correctos (TP): {total_scmb_corr}")
print(f"  > sCMBs Erróneos (FP): {total_scmb_err}")
print("-" * 40)
print(f"Media Recall (Sensibilidad): {df_res['Recall'].mean():.4f}")
print(f"Media Precision: {df_res['Precision'].mean():.4f}")
print(f"Media F1-Score: {df_res['F1-Score'].mean():.4f}")
print(f"Resultados guardados en: {csv_path}")

----------------------------------------
ANÁLISIS FINALIZADO
Total rCMBs (Reales en Excel): 146
Total sCMBs (Generados por el modelo): 231
  > sCMBs Correctos (TP): 52
  > sCMBs Erróneos (FP): 179
----------------------------------------
Media Recall (Sensibilidad): 0.2735
Media Precision: 0.2090
Media F1-Score: 0.2109
Resultados guardados en: /media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results/Dataset112_SyntheticCMB/nnUNet_2D_250epochs_predicciones_test_rCMBs/analisis_predicciones_sobre_rCMBs/analisis_completo_CMB.csv


#### Hacemos limpieza de predicciones menores que 2 vóxeles para ver si mejoran las métricas (se observó que las CMBs detectadas como FP eran en su mayoría píxeles aislados)

In [18]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label, regionprops
from tqdm import tqdm

# ========================= RUTAS =========================
EXCEL_PATH = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/CSIRO_dataset/rCMBInformationInfo.xlsx"
MASK_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results/Dataset112_SyntheticCMB/nnUNet_2D_250epochs_predicciones_test_rCMBs" 
OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results/Dataset112_SyntheticCMB/analisis_final_TF"

MIN_SIZE = 2 # Filtro: Borra lo menor a 2 (es decir, deja >= 2)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========================= FUNCIONES =========================
def safe_int_minus1(val):
    try:
        if pd.isna(val) or str(val).strip() == '' or str(val).strip().lower() == 'nan':
            return None
        return int(float(str(val).strip())) - 1
    except:
        return None

valid_offsets = []
for dx in range(-2, 3):
    for dy in range(-2, 3):
        for dz in range(-2, 3):
            if (dx**2 + dy**2 + dz**2) <= 4:
                valid_offsets.append((dx, dy, dz))

# ========================= PROCESO =========================
df = pd.read_excel(EXCEL_PATH)
audit_results = []

for _, row in tqdm(df.iterrows(), total=df.shape[0]):
    nifti_name = str(row.iloc[0]).strip()
    if not nifti_name.endswith('.nii.gz'): nifti_name += '.nii.gz'
    
    mask_path = os.path.join(MASK_DIR, nifti_name)
    if not os.path.exists(mask_path): continue

    mask_data = nib.load(mask_path).get_fdata()
    shape = mask_data.shape
    
    # 1. Etiquetado
    labeled_mask, num_blobs_orig = label(mask_data > 0, return_num=True)
    props = regionprops(labeled_mask)
    labels_grandes = [p.label for p in props if p.area >= MIN_SIZE]
    num_blobs_filt = len(labels_grandes)

    # 2. Coordenadas Excel
    coords_gt = []
    for i in range(1, len(row), 3):
        x, y, z = safe_int_minus1(row.iloc[i]), safe_int_minus1(row.iloc[i+1]), safe_int_minus1(row.iloc[i+2])
        if all(v is not None for v in [x, y, z]):
            if 0 <= x < shape[0] and 0 <= y < shape[1] and 0 <= z < shape[2]:
                coords_gt.append((x, y, z))

    # 3. Evaluación (Lógica recuperada)
    blobs_hit_orig = set()
    blobs_hit_filt = set()
    gt_encontrados_orig = 0
    gt_encontrados_filt = 0

    for (cx, cy, cz) in coords_gt:
        found_orig = False
        found_filt = False
        for dx, dy, dz in valid_offsets:
            nx, ny, nz = cx + dx, cy + dy, cz + dz
            if 0 <= nx < shape[0] and 0 <= ny < shape[1] and 0 <= nz < shape[2]:
                b_id = labeled_mask[nx, ny, nz]
                if b_id > 0:
                    blobs_hit_orig.add(b_id)
                    found_orig = True
                    if b_id in labels_grandes:
                        blobs_hit_filt.add(b_id)
                        found_filt = True
        
        if found_orig: gt_encontrados_orig += 1
        if found_filt: gt_encontrados_filt += 1

    # --- MÉTRICAS ---
    # RECALL se basa en cuántas rCMB del Excel hemos pillado
    rec_orig = gt_encontrados_orig / len(coords_gt) if coords_gt else 0
    rec_filt = gt_encontrados_filt / len(coords_gt) if coords_gt else 0
    
    # PRECISION se basa en cuántos blobs del modelo eran correctos
    # (Aquí es donde recuperamos tus 52 vs 49)
    prec_orig = len(blobs_hit_orig) / num_blobs_orig if num_blobs_orig > 0 else 0
    prec_filt = len(blobs_hit_filt) / num_blobs_filt if num_blobs_filt > 0 else 0

    audit_results.append({
        'ID': nifti_name,
        'GT_Total': len(coords_gt),
        'Blobs_Correctos_Original': len(blobs_hit_orig),  # Esto sumará 52
        'Blobs_Correctos_Filtrado': len(blobs_hit_filt),
        'GT_Encontrados_Original': gt_encontrados_orig,   # Esto sumará 49
        'GT_Encontrados_Filtrado': gt_encontrados_filt,
        'Num_Blobs_Total': num_blobs_orig,
        'Num_Blobs_Filtrados': num_blobs_filt,
        'Recall_Original': rec_orig,
        'Precision_Original': prec_orig,
        'Recall_Filtrado': rec_filt,
        'Precision_Filtrado': prec_filt
    })

# ========================= SALIDA =========================
df_res = pd.DataFrame(audit_results)
df_res.to_csv(os.path.join(OUTPUT_DIR, "auditoria_final_descriptiva.csv"), index=False)

def f1(p, r): return 2*(p*r)/(p+r) if (p+r)>0 else 0

m_rec_o, m_pre_o = df_res['Recall_Original'].mean(), df_res['Precision_Original'].mean()
m_rec_f, m_pre_f = df_res['Recall_Filtrado'].mean(), df_res['Precision_Filtrado'].mean()

print("\n" + "=".center(60, "="))
print(" COMPARATIVA DE MÉTRICAS (TRIAL 2) ".center(60))
print("=".center(60, "="))
print(f"Total rCMBs (Excel): {df_res['GT_Total'].sum()}")
print(f"Aciertos del Modelo (Blobs TP): {df_res['Blobs_Correctos_Original'].sum()}  --> Tus 52")
print(f"Lesiones Reales Encontradas:    {df_res['GT_Encontrados_Original'].sum()}  --> Tus 49")
print("-" * 60)
print(f"TP TRAS FILTRADO (Blobs):       {df_res['Blobs_Correctos_Filtrado'].sum()}")
print(f"FP ELIMINADOS (Ruido):          {df_res['Num_Blobs_Total'].sum() - df_res['Num_Blobs_Filtrados'].sum()}")
print("-" * 60)
print(f"ORIGINAL -> Rec: {m_rec_o:.4f} | Pre: {m_pre_o:.4f} | F1: {f1(m_pre_o, m_rec_o):.4f}")
print(f"FILTRADO -> Rec: {m_rec_f:.4f} | Pre: {m_pre_f:.4f} | F1: {f1(m_pre_f, m_rec_f):.4f}")
print("=".center(60, "="))

100%|██████████| 57/57 [00:04<00:00, 11.46it/s]


             COMPARATIVA DE MÉTRICAS (TRIAL 2)              
Total rCMBs (Excel): 146
Aciertos del Modelo (Blobs TP): 52  --> Tus 52
Lesiones Reales Encontradas:    49  --> Tus 49
------------------------------------------------------------
TP TRAS FILTRADO (Blobs):       46
FP ELIMINADOS (Ruido):          125
------------------------------------------------------------
ORIGINAL -> Rec: 0.2735 | Pre: 0.2170 | F1: 0.2420
FILTRADO -> Rec: 0.2485 | Pre: 0.2883 | F1: 0.2669


### Dataset 112 3d fullres smallpatch 50 epochs

In [3]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label

# ========================= RUTAS =========================
EXCEL_PATH = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/CSIRO_dataset/rCMBInformationInfo.xlsx"
MASK_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results/Dataset112_SyntheticCMB/nnUNet_3D_fullres_smallpatch_50epochs contra rCMBs/nnUNet_3D_fullres_smallpatch_50epochs_predicciones_test_rCMBs" 
OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results/Dataset112_SyntheticCMB/nnUNet_3D_fullres_smallpatch_50epochs contra rCMBs/analisis_predicciones_sobre_rCMBs/"


os.makedirs(OUTPUT_DIR, exist_ok=True)

def safe_int_minus1(val):
    if pd.isna(val) or str(val).strip() == '': return None
    try:
        raw = int(float(str(val).strip()))
        return raw - 1 if raw >= 0 else None # Ajustado para permitir index 0
    except: return None

# ========================= PROCESAMIENTO =========================
df = pd.read_excel(EXCEL_PATH)
results_list = []
total_rcmbs_real_global = 0

for idx, row in df.iterrows():
    nifti_name = str(row.iloc[0]).strip()
    if not nifti_name.endswith('.nii.gz'): continue

    mask_path = os.path.join(MASK_DIR, nifti_name)
    if not os.path.exists(mask_path): continue

    mask_data = nib.load(mask_path).get_fdata()
    shape = mask_data.shape

    # 1. Extraer coordenadas reales (Ground Truth)
    coords_gt = []
    for i in range(1, len(row), 3):
        x, y, z = safe_int_minus1(row.iloc[i]), safe_int_minus1(row.iloc[i+1]), safe_int_minus1(row.iloc[i+2])
        if all(v is not None for v in [x, y, z]):
            if 0 <= x < shape[0] and 0 <= y < shape[1] and 0 <= z < shape[2]:
                coords_gt.append((x, y, z))
    
    total_rcmbs_real_global += len(coords_gt)

    # 2. Etiquetar detecciones del modelo (sCMBs detectados)
    labeled_mask, num_sCMB_total = label(mask_data > 0, return_num=True)
    
    # 3. Verificación con vecindario Distancia Euclídea <= 2
    
    # [Added by me: Generación precalculada de offsets esféricos para optimizar el bucle]
    # Busca en un rango de -2 a 2 en cada eje, pero solo guarda los vóxeles cuya
    # distancia euclídea al centro sea <= 2.
    valid_offsets = []
    for dx in range(-2, 3):
        for dy in range(-2, 3):
            for dz in range(-2, 3):
                if (dx**2 + dy**2 + dz**2) <= 4:
                    valid_offsets.append((dx, dy, dz))

    ## CAMBIO: contamos TP desde el ground truth para evitar contar doble un TP
    # y no permitir que varios CMBs predichos se asocien al mismo rCMBs

    tp_count = 0
    used_blobs = set() # blobs ya emparejados con algún rCMB
    
    for (cx, cy, cz) in coords_gt:
        candidate_blobs = set()
        
        # Buscamos todos los blobs dentro del radio esférico de 2 vóxeles
        for dx, dy, dz in valid_offsets:
            nx, ny, nz = cx + dx, cy + dy, cz + dz
            
            # Verificar que no nos salimos de los límites de la imagen
            if 0 <= nx < shape[0] and 0 <= ny < shape[1] and 0 <= nz < shape[2]:
                blob_id = labeled_mask[nx, ny, nz]
                if blob_id > 0:
                    candidate_blobs.add(int(blob_id))
                    
        # De los blobs candidatos, nos quedamos con el primero que no esté aún usado
        # greedy matching 1-a-1
        blob_assigned = None
        for blob_id in candidate_blobs:
            if blob_id not in used_blobs:
                blob_assigned = blob_id
                break
        
        if blob_assigned is not None:
            tp_count += 1 # rCMB bien detectado
            used_blobs.add(blob_assigned)
        
    # Cálculos de sCMB (Detecciones del modelo)
    scmb_correctos = len(used_blobs) 
    scmb_erroneos = num_sCMB_total - scmb_correctos
    fn = len(coords_gt) - tp_count

    # Métricas
    # [Added by me: Prevención de división por cero manejada explícitamente en tu código original, la mantengo igual]
    precision = tp_count / num_sCMB_total if num_sCMB_total > 0 else 0
    recall = tp_count / len(coords_gt) if len(coords_gt) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    results_list.append({
        'ID': nifti_name,
        'rCMBs_Reales': len(coords_gt),
        'sCMBs_Detectados_Total': num_sCMB_total,
        'sCMBs_Correctos(TP)': scmb_correctos,
        'sCMBs_Erroneos(FP)': scmb_erroneos,
        'Missed(FN)': fn,
        'Precision': round(precision, 4),
        'Recall': round(recall, 4),
        'F1-Score': round(f1, 4)
    })



# ========================= SALIDA =========================
df_res = pd.DataFrame(results_list)
csv_path = os.path.join(OUTPUT_DIR, "analisis_completo_CMB.csv")
df_res.to_csv(csv_path, index=False)

total_scmb_gen = df_res['sCMBs_Detectados_Total'].sum()
total_scmb_corr = df_res['sCMBs_Correctos(TP)'].sum()
total_scmb_err = df_res['sCMBs_Erroneos(FP)'].sum()

print("-" * 40)
print(f"ANÁLISIS FINALIZADO")
print(f"Total rCMBs (Reales en Excel): {total_rcmbs_real_global}")
print(f"Total sCMBs (Generados por el modelo): {total_scmb_gen}")
print(f"  > sCMBs Correctos (TP): {total_scmb_corr}")
print(f"  > sCMBs Erróneos (FP): {total_scmb_err}")
print("-" * 40)
print(f"Media Recall (Sensibilidad): {df_res['Recall'].mean():.4f}")
print(f"Media Precision: {df_res['Precision'].mean():.4f}")
print(f"Media F1-Score: {df_res['F1-Score'].mean():.4f}")
print(f"Resultados guardados en: {csv_path}")

----------------------------------------
ANÁLISIS FINALIZADO
Total rCMBs (Reales en Excel): 146
Total sCMBs (Generados por el modelo): 302
  > sCMBs Correctos (TP): 103
  > sCMBs Erróneos (FP): 199
----------------------------------------
Media Recall (Sensibilidad): 0.7451
Media Precision: 0.4266
Media F1-Score: 0.5054
Resultados guardados en: /media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results/Dataset112_SyntheticCMB/nnUNet_3D_fullres_smallpatch_50epochs contra rCMBs/analisis_predicciones_sobre_rCMBs/analisis_completo_CMB.csv


### Dataset 112 3d fullres smallpatch 250 epochs

In [4]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label

# ========================= RUTAS =========================
EXCEL_PATH = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/CSIRO_dataset/rCMBInformationInfo.xlsx"
MASK_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results/Dataset112_SyntheticCMB/nnUNet_3D_fullres_smallpatch_250epochs contra rCMBs/nnUNet_3D_fullres_smallpatch_250epochs_predicciones_test_rCMBs" 
OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results/Dataset112_SyntheticCMB/nnUNet_3D_fullres_smallpatch_250epochs contra rCMBs/analisis_predicciones_sobre_rCMBs/"


os.makedirs(OUTPUT_DIR, exist_ok=True)

def safe_int_minus1(val):
    if pd.isna(val) or str(val).strip() == '': return None
    try:
        raw = int(float(str(val).strip()))
        return raw - 1 if raw >= 0 else None # Ajustado para permitir index 0
    except: return None

# ========================= PROCESAMIENTO =========================
df = pd.read_excel(EXCEL_PATH)
results_list = []
total_rcmbs_real_global = 0

for idx, row in df.iterrows():
    nifti_name = str(row.iloc[0]).strip()
    if not nifti_name.endswith('.nii.gz'): continue

    mask_path = os.path.join(MASK_DIR, nifti_name)
    if not os.path.exists(mask_path): continue

    mask_data = nib.load(mask_path).get_fdata()
    shape = mask_data.shape

    # 1. Extraer coordenadas reales (Ground Truth)
    coords_gt = []
    for i in range(1, len(row), 3):
        x, y, z = safe_int_minus1(row.iloc[i]), safe_int_minus1(row.iloc[i+1]), safe_int_minus1(row.iloc[i+2])
        if all(v is not None for v in [x, y, z]):
            if 0 <= x < shape[0] and 0 <= y < shape[1] and 0 <= z < shape[2]:
                coords_gt.append((x, y, z))
    
    total_rcmbs_real_global += len(coords_gt)

    # 2. Etiquetar detecciones del modelo (sCMBs detectados)
    labeled_mask, num_sCMB_total = label(mask_data > 0, return_num=True)
    
    # 3. Verificación con vecindario Distancia Euclídea <= 2
    
    # [Added by me: Generación precalculada de offsets esféricos para optimizar el bucle]
    # Busca en un rango de -2 a 2 en cada eje, pero solo guarda los vóxeles cuya
    # distancia euclídea al centro sea <= 2.
    valid_offsets = []
    for dx in range(-2, 3):
        for dy in range(-2, 3):
            for dz in range(-2, 3):
                if (dx**2 + dy**2 + dz**2) <= 4:
                    valid_offsets.append((dx, dy, dz))

    ## CAMBIO: contamos TP desde el ground truth para evitar contar doble un TP
    # y no permitir que varios CMBs predichos se asocien al mismo rCMBs

    tp_count = 0
    used_blobs = set() # blobs ya emparejados con algún rCMB
    
    for (cx, cy, cz) in coords_gt:
        candidate_blobs = set()
        
        # Buscamos todos los blobs dentro del radio esférico de 2 vóxeles
        for dx, dy, dz in valid_offsets:
            nx, ny, nz = cx + dx, cy + dy, cz + dz
            
            # Verificar que no nos salimos de los límites de la imagen
            if 0 <= nx < shape[0] and 0 <= ny < shape[1] and 0 <= nz < shape[2]:
                blob_id = labeled_mask[nx, ny, nz]
                if blob_id > 0:
                    candidate_blobs.add(int(blob_id))
                    
        # De los blobs candidatos, nos quedamos con el primero que no esté aún usado
        # greedy matching 1-a-1
        blob_assigned = None
        for blob_id in candidate_blobs:
            if blob_id not in used_blobs:
                blob_assigned = blob_id
                break
        
        if blob_assigned is not None:
            tp_count += 1 # rCMB bien detectado
            used_blobs.add(blob_assigned)
        
    # Cálculos de sCMB (Detecciones del modelo)
    scmb_correctos = len(used_blobs) 
    scmb_erroneos = num_sCMB_total - scmb_correctos
    fn = len(coords_gt) - tp_count

    # Métricas
    # [Added by me: Prevención de división por cero manejada explícitamente en tu código original, la mantengo igual]
    precision = tp_count / num_sCMB_total if num_sCMB_total > 0 else 0
    recall = tp_count / len(coords_gt) if len(coords_gt) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    results_list.append({
        'ID': nifti_name,
        'rCMBs_Reales': len(coords_gt),
        'sCMBs_Detectados_Total': num_sCMB_total,
        'sCMBs_Correctos(TP)': scmb_correctos,
        'sCMBs_Erroneos(FP)': scmb_erroneos,
        'Missed(FN)': fn,
        'Precision': round(precision, 4),
        'Recall': round(recall, 4),
        'F1-Score': round(f1, 4)
    })



# ========================= SALIDA =========================
df_res = pd.DataFrame(results_list)
csv_path = os.path.join(OUTPUT_DIR, "analisis_completo_CMB.csv")
df_res.to_csv(csv_path, index=False)

total_scmb_gen = df_res['sCMBs_Detectados_Total'].sum()
total_scmb_corr = df_res['sCMBs_Correctos(TP)'].sum()
total_scmb_err = df_res['sCMBs_Erroneos(FP)'].sum()

print("-" * 40)
print(f"ANÁLISIS FINALIZADO")
print(f"Total rCMBs (Reales en Excel): {total_rcmbs_real_global}")
print(f"Total sCMBs (Generados por el modelo): {total_scmb_gen}")
print(f"  > sCMBs Correctos (TP): {total_scmb_corr}")
print(f"  > sCMBs Erróneos (FP): {total_scmb_err}")
print("-" * 40)
print(f"Media Recall (Sensibilidad): {df_res['Recall'].mean():.4f}")
print(f"Media Precision: {df_res['Precision'].mean():.4f}")
print(f"Media F1-Score: {df_res['F1-Score'].mean():.4f}")
print(f"Resultados guardados en: {csv_path}")

----------------------------------------
ANÁLISIS FINALIZADO
Total rCMBs (Reales en Excel): 146
Total sCMBs (Generados por el modelo): 100
  > sCMBs Correctos (TP): 25
  > sCMBs Erróneos (FP): 75
----------------------------------------
Media Recall (Sensibilidad): 0.1679
Media Precision: 0.2126
Media F1-Score: 0.1592
Resultados guardados en: /media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results/Dataset112_SyntheticCMB/nnUNet_3D_fullres_smallpatch_250epochs contra rCMBs/analisis_predicciones_sobre_rCMBs/analisis_completo_CMB.csv


### Dataset 113 3d fullres smallpatch 250 epochs

In [1]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label

# ========================= RUTAS =========================
EXCEL_PATH = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/CSIRO_dataset/rCMBInformationInfo.xlsx"
MASK_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results/Dataset113_SyntheticCMB/nnUNet_3D_fullres_smallpatch_250epochs contra rCMBs/nnUNet_3D_fullres_smallpatch_250epochs_predicciones_test_rCMBs" 
OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results/Dataset113_SyntheticCMB/nnUNet_3D_fullres_smallpatch_250epochs contra rCMBs/analisis_predicciones_sobre_rCMBs/"


os.makedirs(OUTPUT_DIR, exist_ok=True)

def safe_int_minus1(val):
    if pd.isna(val) or str(val).strip() == '': return None
    try:
        raw = int(float(str(val).strip()))
        return raw - 1 if raw >= 0 else None # Ajustado para permitir index 0
    except: return None

# ========================= PROCESAMIENTO =========================
df = pd.read_excel(EXCEL_PATH)
results_list = []
total_rcmbs_real_global = 0

for idx, row in df.iterrows():
    nifti_name = str(row.iloc[0]).strip()
    if not nifti_name.endswith('.nii.gz'): continue

    mask_path = os.path.join(MASK_DIR, nifti_name)
    if not os.path.exists(mask_path): continue

    mask_data = nib.load(mask_path).get_fdata()
    shape = mask_data.shape

    # 1. Extraer coordenadas reales (Ground Truth)
    coords_gt = []
    for i in range(1, len(row), 3):
        x, y, z = safe_int_minus1(row.iloc[i]), safe_int_minus1(row.iloc[i+1]), safe_int_minus1(row.iloc[i+2])
        if all(v is not None for v in [x, y, z]):
            if 0 <= x < shape[0] and 0 <= y < shape[1] and 0 <= z < shape[2]:
                coords_gt.append((x, y, z))
    
    total_rcmbs_real_global += len(coords_gt)

    # 2. Etiquetar detecciones del modelo (sCMBs detectados)
    labeled_mask, num_sCMB_total = label(mask_data > 0, return_num=True)
    
    # 3. Verificación con vecindario Distancia Euclídea <= 2
    
    # [Added by me: Generación precalculada de offsets esféricos para optimizar el bucle]
    # Busca en un rango de -2 a 2 en cada eje, pero solo guarda los vóxeles cuya
    # distancia euclídea al centro sea <= 2.
    valid_offsets = []
    for dx in range(-2, 3):
        for dy in range(-2, 3):
            for dz in range(-2, 3):
                if (dx**2 + dy**2 + dz**2) <= 4:
                    valid_offsets.append((dx, dy, dz))

    ## CAMBIO: contamos TP desde el ground truth para evitar contar doble un TP
    # y no permitir que varios CMBs predichos se asocien al mismo rCMBs

    tp_count = 0
    used_blobs = set() # blobs ya emparejados con algún rCMB
    
    for (cx, cy, cz) in coords_gt:
        candidate_blobs = set()
        
        # Buscamos todos los blobs dentro del radio esférico de 2 vóxeles
        for dx, dy, dz in valid_offsets:
            nx, ny, nz = cx + dx, cy + dy, cz + dz
            
            # Verificar que no nos salimos de los límites de la imagen
            if 0 <= nx < shape[0] and 0 <= ny < shape[1] and 0 <= nz < shape[2]:
                blob_id = labeled_mask[nx, ny, nz]
                if blob_id > 0:
                    candidate_blobs.add(int(blob_id))
                    
        # De los blobs candidatos, nos quedamos con el primero que no esté aún usado
        # greedy matching 1-a-1
        blob_assigned = None
        for blob_id in candidate_blobs:
            if blob_id not in used_blobs:
                blob_assigned = blob_id
                break
        
        if blob_assigned is not None:
            tp_count += 1 # rCMB bien detectado
            used_blobs.add(blob_assigned)
        
    # Cálculos de sCMB (Detecciones del modelo)
    scmb_correctos = len(used_blobs) 
    scmb_erroneos = num_sCMB_total - scmb_correctos
    fn = len(coords_gt) - tp_count

    # Métricas
    # [Added by me: Prevención de división por cero manejada explícitamente en tu código original, la mantengo igual]
    precision = tp_count / num_sCMB_total if num_sCMB_total > 0 else 0
    recall = tp_count / len(coords_gt) if len(coords_gt) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    results_list.append({
        'ID': nifti_name,
        'rCMBs_Reales': len(coords_gt),
        'sCMBs_Detectados_Total': num_sCMB_total,
        'sCMBs_Correctos(TP)': scmb_correctos,
        'sCMBs_Erroneos(FP)': scmb_erroneos,
        'Missed(FN)': fn,
        'Precision': round(precision, 4),
        'Recall': round(recall, 4),
        'F1-Score': round(f1, 4)
    })



# ========================= SALIDA =========================
df_res = pd.DataFrame(results_list)
csv_path = os.path.join(OUTPUT_DIR, "analisis_completo_CMB.csv")
df_res.to_csv(csv_path, index=False)

total_scmb_gen = df_res['sCMBs_Detectados_Total'].sum()
total_scmb_corr = df_res['sCMBs_Correctos(TP)'].sum()
total_scmb_err = df_res['sCMBs_Erroneos(FP)'].sum()

print("-" * 40)
print(f"ANÁLISIS FINALIZADO")
print(f"Total rCMBs (Reales en Excel): {total_rcmbs_real_global}")
print(f"Total sCMBs (Generados por el modelo): {total_scmb_gen}")
print(f"  > sCMBs Correctos (TP): {total_scmb_corr}")
print(f"  > sCMBs Erróneos (FP): {total_scmb_err}")
print("-" * 40)
print(f"Media Recall (Sensibilidad): {df_res['Recall'].mean():.4f}")
print(f"Media Precision: {df_res['Precision'].mean():.4f}")
print(f"Media F1-Score: {df_res['F1-Score'].mean():.4f}")
print(f"Resultados guardados en: {csv_path}")

----------------------------------------
ANÁLISIS FINALIZADO
Total rCMBs (Reales en Excel): 146
Total sCMBs (Generados por el modelo): 152
  > sCMBs Correctos (TP): 38
  > sCMBs Erróneos (FP): 114
----------------------------------------
Media Recall (Sensibilidad): 0.2946
Media Precision: 0.1960
Media F1-Score: 0.2040
Resultados guardados en: /media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results/Dataset113_SyntheticCMB/nnUNet_3D_fullres_smallpatch_250epochs contra rCMBs/analisis_predicciones_sobre_rCMBs/analisis_completo_CMB.csv


### FINE TUNING: Dataset 115 3d fullres smallpatch 50 epochs en rCMBs con pseudomáscaras
### Procedente de checkpoint best D112 3d fullres smallpatch 50 epochs (alta sensibilidad, baja precisión)

In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label

# ========================= RUTAS =========================
EXCEL_PATH = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/CSIRO_dataset/rCMBInformationInfo.xlsx"
MASK_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results/Dataset115_RealFineTuning/prediction_results"
OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results/Dataset115_RealFineTuning/analysis_results"


os.makedirs(OUTPUT_DIR, exist_ok=True)

def safe_int_minus1(val):
    if pd.isna(val) or str(val).strip() == '': return None
    try:
        raw = int(float(str(val).strip()))
        return raw - 1 if raw >= 0 else None # Ajustado para permitir index 0
    except: return None

# ========================= PROCESAMIENTO =========================
df = pd.read_excel(EXCEL_PATH)
results_list = []
total_rcmbs_real_global = 0

for idx, row in df.iterrows():
    nifti_name = str(row.iloc[0]).strip()
    if not nifti_name.endswith('.nii.gz'): continue

    mask_path = os.path.join(MASK_DIR, nifti_name)
    if not os.path.exists(mask_path): continue

    mask_data = nib.load(mask_path).get_fdata()
    shape = mask_data.shape

    # 1. Extraer coordenadas reales (Ground Truth)
    coords_gt = []
    for i in range(1, len(row), 3):
        x, y, z = safe_int_minus1(row.iloc[i]), safe_int_minus1(row.iloc[i+1]), safe_int_minus1(row.iloc[i+2])
        if all(v is not None for v in [x, y, z]):
            if 0 <= x < shape[0] and 0 <= y < shape[1] and 0 <= z < shape[2]:
                coords_gt.append((x, y, z))
    
    total_rcmbs_real_global += len(coords_gt)

    # 2. Etiquetar detecciones del modelo (sCMBs detectados)
    labeled_mask, num_sCMB_total = label(mask_data > 0, return_num=True)
    
    # 3. Verificación con vecindario Distancia Euclídea <= 2
    
    # [Added by me: Generación precalculada de offsets esféricos para optimizar el bucle]
    # Busca en un rango de -2 a 2 en cada eje, pero solo guarda los vóxeles cuya
    # distancia euclídea al centro sea <= 2.
    valid_offsets = []
    for dx in range(-2, 3):
        for dy in range(-2, 3):
            for dz in range(-2, 3):
                if (dx**2 + dy**2 + dz**2) <= 4:
                    valid_offsets.append((dx, dy, dz))

    ## CAMBIO: contamos TP desde el ground truth para evitar contar doble un TP
    # y no permitir que varios CMBs predichos se asocien al mismo rCMBs

    tp_count = 0
    used_blobs = set() # blobs ya emparejados con algún rCMB
    
    for (cx, cy, cz) in coords_gt:
        candidate_blobs = set()
        
        # Buscamos todos los blobs dentro del radio esférico de 2 vóxeles
        for dx, dy, dz in valid_offsets:
            nx, ny, nz = cx + dx, cy + dy, cz + dz
            
            # Verificar que no nos salimos de los límites de la imagen
            if 0 <= nx < shape[0] and 0 <= ny < shape[1] and 0 <= nz < shape[2]:
                blob_id = labeled_mask[nx, ny, nz]
                if blob_id > 0:
                    candidate_blobs.add(int(blob_id))
                    
        # De los blobs candidatos, nos quedamos con el primero que no esté aún usado
        # greedy matching 1-a-1
        blob_assigned = None
        for blob_id in candidate_blobs:
            if blob_id not in used_blobs:
                blob_assigned = blob_id
                break
        
        if blob_assigned is not None:
            tp_count += 1 # rCMB bien detectado
            used_blobs.add(blob_assigned)
        
    # Cálculos de sCMB (Detecciones del modelo)
    scmb_correctos = len(used_blobs) 
    scmb_erroneos = num_sCMB_total - scmb_correctos
    fn = len(coords_gt) - tp_count

    # Métricas
    # [Added by me: Prevención de división por cero manejada explícitamente en tu código original, la mantengo igual]
    precision = tp_count / num_sCMB_total if num_sCMB_total > 0 else 0
    recall = tp_count / len(coords_gt) if len(coords_gt) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    results_list.append({
        'ID': nifti_name,
        'rCMBs_Reales': len(coords_gt),
        'sCMBs_Detectados_Total': num_sCMB_total,
        'sCMBs_Correctos(TP)': scmb_correctos,
        'sCMBs_Erroneos(FP)': scmb_erroneos,
        'Missed(FN)': fn,
        'Precision': round(precision, 4),
        'Recall': round(recall, 4),
        'F1-Score': round(f1, 4)
    })



# ========================= SALIDA =========================
df_res = pd.DataFrame(results_list)
csv_path = os.path.join(OUTPUT_DIR, "analisis_completo_CMB.csv")
df_res.to_csv(csv_path, index=False)

total_scmb_gen = df_res['sCMBs_Detectados_Total'].sum()
total_scmb_corr = df_res['sCMBs_Correctos(TP)'].sum()
total_scmb_err = df_res['sCMBs_Erroneos(FP)'].sum()

print("-" * 40)
print(f"ANÁLISIS FINALIZADO")
print(f"Total rCMBs (Reales en Excel): {total_rcmbs_real_global}")
print(f"Total sCMBs (Generados por el modelo): {total_scmb_gen}")
print(f"  > sCMBs Correctos (TP): {total_scmb_corr}")
print(f"  > sCMBs Erróneos (FP): {total_scmb_err}")
print("-" * 40)
print(f"Media Recall (Sensibilidad): {df_res['Recall'].mean():.4f}")
print(f"Media Precision: {df_res['Precision'].mean():.4f}")
print(f"Media F1-Score: {df_res['F1-Score'].mean():.4f}")
print(f"Resultados guardados en: {csv_path}")

----------------------------------------
ANÁLISIS FINALIZADO
Total rCMBs (Reales en Excel): 32
Total sCMBs (Generados por el modelo): 26
  > sCMBs Correctos (TP): 20
  > sCMBs Erróneos (FP): 6
----------------------------------------
Media Recall (Sensibilidad): 0.7178
Media Precision: 0.7278
Media F1-Score: 0.7022
Resultados guardados en: /media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results/Dataset115_RealFineTuning/analysis_results/analisis_completo_CMB.csv


### ONLY REAL: Dataset 116 3d fullres smallpatch 50 epochs en rCMBs con pseudomáscaras

In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label

# ========================= RUTAS =========================
EXCEL_PATH = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/CSIRO_dataset/rCMBInformationInfo.xlsx"
MASK_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results/Dataset116_RealCMB/predicts_test_interno"
OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results/Dataset116_RealCMB/analysis_results"


os.makedirs(OUTPUT_DIR, exist_ok=True)

def safe_int_minus1(val):
    if pd.isna(val) or str(val).strip() == '': return None
    try:
        raw = int(float(str(val).strip()))
        return raw - 1 if raw >= 0 else None # Ajustado para permitir index 0
    except: return None

# ========================= PROCESAMIENTO =========================
df = pd.read_excel(EXCEL_PATH)
results_list = []
total_rcmbs_real_global = 0

for idx, row in df.iterrows():
    nifti_name = str(row.iloc[0]).strip()
    if not nifti_name.endswith('.nii.gz'): continue

    mask_path = os.path.join(MASK_DIR, nifti_name)
    if not os.path.exists(mask_path): continue

    mask_data = nib.load(mask_path).get_fdata()
    shape = mask_data.shape

    # 1. Extraer coordenadas reales (Ground Truth)
    coords_gt = []
    for i in range(1, len(row), 3):
        x, y, z = safe_int_minus1(row.iloc[i]), safe_int_minus1(row.iloc[i+1]), safe_int_minus1(row.iloc[i+2])
        if all(v is not None for v in [x, y, z]):
            if 0 <= x < shape[0] and 0 <= y < shape[1] and 0 <= z < shape[2]:
                coords_gt.append((x, y, z))
    
    total_rcmbs_real_global += len(coords_gt)

    # 2. Etiquetar detecciones del modelo (sCMBs detectados)
    labeled_mask, num_sCMB_total = label(mask_data > 0, return_num=True)
    
    # 3. Verificación con vecindario Distancia Euclídea <= 2
    
    # [Added by me: Generación precalculada de offsets esféricos para optimizar el bucle]
    # Busca en un rango de -2 a 2 en cada eje, pero solo guarda los vóxeles cuya
    # distancia euclídea al centro sea <= 2.
    valid_offsets = []
    for dx in range(-2, 3):
        for dy in range(-2, 3):
            for dz in range(-2, 3):
                if (dx**2 + dy**2 + dz**2) <= 4:
                    valid_offsets.append((dx, dy, dz))

    ## CAMBIO: contamos TP desde el ground truth para evitar contar doble un TP
    # y no permitir que varios CMBs predichos se asocien al mismo rCMBs

    tp_count = 0
    used_blobs = set() # blobs ya emparejados con algún rCMB
    
    for (cx, cy, cz) in coords_gt:
        candidate_blobs = set()
        
        # Buscamos todos los blobs dentro del radio esférico de 2 vóxeles
        for dx, dy, dz in valid_offsets:
            nx, ny, nz = cx + dx, cy + dy, cz + dz
            
            # Verificar que no nos salimos de los límites de la imagen
            if 0 <= nx < shape[0] and 0 <= ny < shape[1] and 0 <= nz < shape[2]:
                blob_id = labeled_mask[nx, ny, nz]
                if blob_id > 0:
                    candidate_blobs.add(int(blob_id))
                    
        # De los blobs candidatos, nos quedamos con el primero que no esté aún usado
        # greedy matching 1-a-1
        blob_assigned = None
        for blob_id in candidate_blobs:
            if blob_id not in used_blobs:
                blob_assigned = blob_id
                break
        
        if blob_assigned is not None:
            tp_count += 1 # rCMB bien detectado
            used_blobs.add(blob_assigned)
        
    # Cálculos de sCMB (Detecciones del modelo)
    scmb_correctos = len(used_blobs) 
    scmb_erroneos = num_sCMB_total - scmb_correctos
    fn = len(coords_gt) - tp_count

    # Métricas
    # [Added by me: Prevención de división por cero manejada explícitamente en tu código original, la mantengo igual]
    precision = tp_count / num_sCMB_total if num_sCMB_total > 0 else 0
    recall = tp_count / len(coords_gt) if len(coords_gt) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    results_list.append({
        'ID': nifti_name,
        'rCMBs_Reales': len(coords_gt),
        'sCMBs_Detectados_Total': num_sCMB_total,
        'sCMBs_Correctos(TP)': scmb_correctos,
        'sCMBs_Erroneos(FP)': scmb_erroneos,
        'Missed(FN)': fn,
        'Precision': round(precision, 4),
        'Recall': round(recall, 4),
        'F1-Score': round(f1, 4)
    })



# ========================= SALIDA =========================
df_res = pd.DataFrame(results_list)
csv_path = os.path.join(OUTPUT_DIR, "analisis_completo_CMB.csv")
df_res.to_csv(csv_path, index=False)

total_scmb_gen = df_res['sCMBs_Detectados_Total'].sum()
total_scmb_corr = df_res['sCMBs_Correctos(TP)'].sum()
total_scmb_err = df_res['sCMBs_Erroneos(FP)'].sum()

print("-" * 40)
print(f"ANÁLISIS FINALIZADO")
print(f"Total rCMBs (Reales en Excel): {total_rcmbs_real_global}")
print(f"Total sCMBs (Generados por el modelo): {total_scmb_gen}")
print(f"  > sCMBs Correctos (TP): {total_scmb_corr}")
print(f"  > sCMBs Erróneos (FP): {total_scmb_err}")
print("-" * 40)
print(f"Media Recall (Sensibilidad): {df_res['Recall'].mean():.4f}")
print(f"Media Precision: {df_res['Precision'].mean():.4f}")
print(f"Media F1-Score: {df_res['F1-Score'].mean():.4f}")
print(f"Resultados guardados en: {csv_path}")

----------------------------------------
ANÁLISIS FINALIZADO
Total rCMBs (Reales en Excel): 32
Total sCMBs (Generados por el modelo): 26
  > sCMBs Correctos (TP): 20
  > sCMBs Erróneos (FP): 6
----------------------------------------
Media Recall (Sensibilidad): 0.7044
Media Precision: 0.7333
Media F1-Score: 0.6981
Resultados guardados en: /media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results/Dataset116_RealCMB/analysis_results/analisis_completo_CMB.csv
